# Baseline PAL — uniform sampling

Public reproducibility version for the manuscript *Mapping-Class-Group Learning for Diffeomorphisms*.

**Important:** the revised manuscript reports candidate selection by unpenalized validation loss (equivalently, `lambda = 0`). The original exploratory notebooks also logged a reference word-length penalty of `1e-7`; that legacy setting did not change the selected candidate in the reported runs.

Run Jupyter from the repository root so that relative output paths under `results/` resolve as intended.


In [ ]:
# ============================================================
# PAL residual learner for mapping-class puzzle learning
# Uniform sampling on T^2
# with train / validation / test split
# ============================================================

import os
import random
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 0. Reproducibility setting
# -----------------------------

SEED = 0

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print("Could not enforce deterministic algorithms:", e)

device = torch.device("cpu")
print("device:", device)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)


# -----------------------------
# 1. Torus utilities
# -----------------------------

def torus_diff(pred, target):
    """
    Shortest coordinate-wise difference on T^2.
    Both pred and target are tensors of shape [N, 2].
    """
    diff = pred - target
    diff = (diff + 0.5) % 1.0 - 0.5
    return diff

def torus_mse(pred, target):
    """
    Mean squared error on T^2 using shortest periodic differences.
    """
    return torch.mean(torus_diff(pred, target) ** 2)


# -----------------------------
# 2. Mapping-class generators
# -----------------------------

def A_power(k):
    """
    A^k(x1,x2) = (x1 + k x2, x2) mod 1.
    """
    def G(x):
        u = (x[..., 0] + k * x[..., 1]) % 1.0
        v = x[..., 1]
        return torch.stack([u, v], dim=-1)
    return G

def B_power(k):
    """
    B^k(x1,x2) = (x1, k x1 + x2) mod 1.
    """
    def G(x):
        u = x[..., 0]
        v = (k * x[..., 0] + x[..., 1]) % 1.0
        return torch.stack([u, v], dim=-1)
    return G

def compose(G, H):
    """
    Return G o H.
    """
    def GH(x):
        return G(H(x))
    return GH

A_map = A_power(1)
B_map = B_power(1)

AB_map = compose(A_map, B_map)  # A o B
BA_map = compose(B_map, A_map)  # B o A


# -----------------------------
# 3. Target map f = F0 o AB
# -----------------------------

def F0(x, eps=0.05):
    """
    Small identity-component deformation on T^2.
    """
    u = (x[..., 0] + eps * torch.sin(2 * torch.pi * (x[..., 0] - x[..., 1]))) % 1.0
    v = (x[..., 1] + eps * torch.sin(2 * torch.pi * x[..., 1])) % 1.0
    return torch.stack([u, v], dim=-1)

def f_target_AB(x, eps=0.05):
    """
    Target map f = F0 o AB.
    The hidden mapping-class word is AB.
    """
    return F0(AB_map(x), eps=eps)


# -----------------------------
# 4. Train / validation / test data
# -----------------------------

N_train = 2000
N_val   = 1000
N_test  = 1000

# Important: these are independent samples from Unif(T^2)
x_train = torch.rand(N_train, 2, device=device)
x_val   = torch.rand(N_val, 2, device=device)
x_test  = torch.rand(N_test, 2, device=device)

y_train = f_target_AB(x_train)
y_val   = f_target_AB(x_val)
y_test  = f_target_AB(x_test)

print("x_train:", x_train.shape)
print("x_val:  ", x_val.shape)
print("x_test: ", x_test.shape)


# -----------------------------
# 5. Candidate mapping-class words
# -----------------------------

candidate_G = {
    "id": {
        "map": A_power(0),
        "length": 0
    },
    "A": {
        "map": A_power(1),
        "length": 1
    },
    "A^-1": {
        "map": A_power(-1),
        "length": 1
    },
    "B": {
        "map": B_power(1),
        "length": 1
    },
    "B^-1": {
        "map": B_power(-1),
        "length": 1
    },
    "AB": {
        "map": AB_map,
        "length": 2
    },
    "BA": {
        "map": BA_map,
        "length": 2
    },
    "A^2": {
        "map": A_power(2),
        "length": 2
    },
    "B^2": {
        "map": B_power(2),
        "length": 2
    }
}


# -----------------------------
# 6. PAL residual learner
# -----------------------------

class PALResidualTorus(torch.nn.Module):
    """
    Parametric autonomous ODE residual learner on T^2 x S^1.

    The model defines a residual map R_theta:T^2 -> T^2
    as the T^2-component of the time-one flow.
    """

    def __init__(self, ks=None, D=2, hidden_scale=0.1, steps=40):
        super().__init__()

        if ks is None:
            ks = [
                (0, 0),
                (1, 0), (0, 1),
                (1, 1), (1, -1),
                (2, 0), (0, 2)
            ]

        self.register_buffer("ks", torch.tensor(ks, dtype=torch.float32))
        self.D = D
        self.steps = steps
        self.hidden_scale = hidden_scale

        # oscillator frequency: one full rotation over t in [0,1]
        self.omega = 2 * torch.pi

        # polynomial basis in s=(s1,s2), degree <= D
        self.pq = [(p, q) for p in range(D + 1) for q in range(D + 1 - p)]

        M = len(ks)

        # For each k, use cos(2pi k.x) and sin(2pi k.x)
        self.num_space_features = 2 * M
        self.num_time_features = len(self.pq)

        # theta[space_feature, time_feature, output_dim]
        self.theta = torch.nn.Parameter(
            0.01 * torch.randn(self.num_space_features, self.num_time_features, 2)
        )

    def time_features(self, s):
        """
        Polynomial features of oscillator state s=(s1,s2).
        """
        s1 = s[:, 0]
        s2 = s[:, 1]

        feats = []
        for p, q in self.pq:
            feats.append((s1 ** p) * (s2 ** q))

        return torch.stack(feats, dim=-1)

    def space_features(self, x):
        """
        Fourier features on T^2.
        """
        phase = 2 * torch.pi * (x @ self.ks.T)

        c = torch.cos(phase)
        s = torch.sin(phase)

        return torch.cat([c, s], dim=-1)

    def vector_field_x(self, x, s):
        """
        dx/dt = sum_{m,q} theta_{m,q} phi_m(x) P_q(s).
        """
        phi = self.space_features(x)
        P = self.time_features(s)

        dx = torch.einsum("bp,bq,pqd->bd", phi, P, self.theta)

        return self.hidden_scale * dx

    def rhs(self, z):
        """
        Autonomous RHS on T^2 x S^1.
        """
        x = z[:, :2]
        s = z[:, 2:]

        dx = self.vector_field_x(x, s)

        # ds/dt = omega J s, J(s1,s2)=(-s2,s1)
        ds = self.omega * torch.stack([-s[:, 1], s[:, 0]], dim=-1)

        return torch.cat([dx, ds], dim=-1)

    def rk4_step(self, z, dt):
        k1 = self.rhs(z)
        k2 = self.rhs(z + 0.5 * dt * k1)
        k3 = self.rhs(z + 0.5 * dt * k2)
        k4 = self.rhs(z + dt * k3)
        return z + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

    def forward(self, x):
        batch = x.shape[0]

        # initial oscillator state s=(1,0)
        s0 = torch.zeros(batch, 2, device=x.device)
        s0[:, 0] = 1.0

        z = torch.cat([x, s0], dim=-1)

        dt = 1.0 / self.steps

        for _ in range(self.steps):
            z = self.rk4_step(z, dt)

        xT = z[:, :2] % 1.0

        return xT


# -----------------------------
# 7. Training function
# -----------------------------

def train_candidate_pal(
    G_map,
    x_train,
    y_train,
    x_val,
    y_val,
    D=2,
    steps=40,
    epochs=1000,
    lr=1e-3,
    seed=0,
    hidden_scale=0.1,
    checkpoint_interval=10
):
    """
    Train PAL residual learner for a fixed candidate G.

    Returns:
        model: trained PAL model
        train_losses: training losses at checkpoints
        val_losses: validation losses at checkpoints
    """

    torch.manual_seed(seed)

    model = PALResidualTorus(D=D, steps=steps, hidden_scale=hidden_scale).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        opt.zero_grad()

        gx_train = G_map(x_train)
        pred_train = model(gx_train)

        train_loss = torus_mse(pred_train, y_train)
        train_loss.backward()
        opt.step()

        if epoch % checkpoint_interval == 0 or epoch == epochs - 1:
            model.eval()
            with torch.no_grad():
                gx_val = G_map(x_val)
                pred_val = model(gx_val)
                val_loss = torus_mse(pred_val, y_val)

            train_losses.append(train_loss.item())
            val_losses.append(val_loss.item())

    return model, train_losses, val_losses


# -----------------------------
# 8. Candidate search using validation set
# -----------------------------

# Revised manuscript: primary selector is unpenalized validation loss.

lambda_word = 0.0
results_pal = {}

for name, info in candidate_G.items():
    print(f"Training PAL candidate G = {name}")

    model, train_losses, val_losses = train_candidate_pal(
        info["map"],
        x_train,
        y_train,
        x_val,
        y_val,
        D=2,
        steps=40,
        epochs=1000,
        lr=1e-3,
        seed=SEED,
        hidden_scale=0.1,
        checkpoint_interval=10
    )

    final_val_loss = val_losses[-1]
    score = final_val_loss + lambda_word * info["length"]

    results_pal[name] = {
        "model": model,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "final_val_loss": final_val_loss,
        "word_length": info["length"],
        "score": score
    }

    print(f"  final validation loss = {final_val_loss:.6e}")
    print(f"  word length           = {info['length']}")
    print(f"  selection score       = {score:.6e}")


# -----------------------------
# 9. Show validation results
# -----------------------------

summary_rows = []

for name, res in results_pal.items():
    summary_rows.append({
        "candidate": name,
        "validation_loss": res["final_val_loss"],
        "word_length": res["word_length"],
        "score": res["score"]
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("score").reset_index(drop=True)

display(summary_df)

best_row = summary_df.iloc[0]
best_name = best_row["candidate"]

print("Selected word:", best_name)
print("Best validation loss:", best_row["validation_loss"])
print("Best selection score:", best_row["score"])


# -----------------------------
# 10. Final test evaluation
# -----------------------------

best_model = results_pal[best_name]["model"]
best_G_map = candidate_G[best_name]["map"]

best_model.eval()

with torch.no_grad():
    gx_test = best_G_map(x_test)
    pred_test = best_model(gx_test)
    final_test_loss = torus_mse(pred_test, y_test).item()

print("Final test result")
print("-----------------")
print("Selected word:", best_name)
print(f"Test torus MSE: {final_test_loss:.6e}")


# -----------------------------
# 11. Optional: plot loss curves
# -----------------------------

plt.figure(figsize=(7, 5))

for name, res in results_pal.items():
    plt.plot(res["val_losses"], label=name)

plt.yscale("log")
plt.xlabel("checkpoint")
plt.ylabel("validation torus MSE")
plt.title("PAL validation loss for each candidate word")
plt.legend()
plt.grid(True)
plt.show()